# PROCESADO DE IMÁGENES FACEFORENSICS++ C0 (SIN COMPRESIÓN / RAW)

Extrae caras con MTCNN de los vídeos C0 descargados (50 reales + 50 fakes).
Genera 15 frames por vídeo → ~750 imágenes reales + ~750 imágenes fake.

In [1]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Trabajando con: {device}")

Trabajando con: cuda


## Diagnóstico: vemos qué carpetas e imágenes tenemos

In [2]:
import os

# Ruta donde descargaste los vídeos C0 con el script oficial
BASE_PATH = r'C:\TFG\Dataset_C0'

def diagnostico_profundo(ruta):
    print(f"🔍 Analizando: {ruta}")
    if not os.path.exists(ruta):
        print("❌ ERROR: La ruta NO existe para Python.")
        return

    items = os.listdir(ruta)
    print(f"📁 Carpetas encontradas en la base: {items}")

    for root, dirs, files in os.walk(ruta):
        videos = [f for f in files if f.lower().endswith(('.mp4', '.avi', '.mkv'))]
        if videos:
            nivel = root.replace(ruta, "")
            print(f"\n✅ Carpeta con vídeos: ...{nivel}")
            print(f"   Cant. vídeos: {len(videos)}")
            print(f"   Ejemplo: '{videos[0]}'")
            ejemplo = videos[0][:3]
            if ejemplo.isdigit():
                print(f"   🔢 Nombre empieza por número: SÍ ({ejemplo})")
            else:
                print(f"   ❓ Nombre empieza por: '{ejemplo}'")

diagnostico_profundo(BASE_PATH)

🔍 Analizando: C:\TFG\Dataset_C0
📁 Carpetas encontradas en la base: ['download-FaceForensics.py', 'manipulated_sequences', 'original_sequences']

✅ Carpeta con vídeos: ...\manipulated_sequences\Deepfakes\raw\videos
   Cant. vídeos: 200
   Ejemplo: '004_982.mp4'
   🔢 Nombre empieza por número: SÍ (004)

✅ Carpeta con vídeos: ...\original_sequences\youtube\raw\videos
   Cant. vídeos: 200
   Ejemplo: '004.mp4'
   🔢 Nombre empieza por número: SÍ (004)


## Extracción de caras con MTCNN

In [4]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from facenet_pytorch import MTCNN
from tqdm import tqdm

# ==========================================
# 1. CONFIGURACIÓN DE RUTAS
# ==========================================
# Estructura que deja el script oficial de descarga:
#   C:\TFG\Dataset_C0\original_sequences\youtube\raw\videos\       ← reales
#   C:\TFG\Dataset_C0\manipulated_sequences\Deepfakes\raw\videos\ ← fakes

BASE_PATH    = r'C:\TFG\Dataset_C0'
DESTINY_PATH = r'C:\TFG\Dataset_C0_Procesado'

FRAMES_PER_VIDEO = 15
IMG_SIZE = 299

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Dispositivo: {device}")

mtcnn = MTCNN(image_size=IMG_SIZE, margin=30, keep_all=False, post_process=False, device=device)

# ==========================================
# 2. GENERAR PLAN BALANCEADO
# ==========================================
def generar_plan_balanceado():
    instrucciones = []

    # --- REALES: script oficial descarga en original_sequences/youtube/raw/videos ---
    path_real = os.path.join(BASE_PATH, 'original_sequences', 'youtube', 'raw', 'videos')
    if os.path.exists(path_real):
        archivos_real = sorted([f for f in os.listdir(path_real) if f.endswith('.mp4')])
        print(f"📂 Reales encontrados: {len(archivos_real)} vídeos en {path_real}")
        for f in archivos_real:
            v_id = os.path.splitext(f)[0]  # ej: '000'
            instrucciones.append({
                'path': os.path.join(path_real, f),
                'label': 'real',
                'tech': 'original',
                'id': v_id
            })
    else:
        print(f"❌ No encontrada carpeta de reales: {path_real}")

    # --- FAKES: script oficial descarga en manipulated_sequences/Deepfakes/raw/videos ---
    path_fake = os.path.join(BASE_PATH, 'manipulated_sequences', 'Deepfakes', 'raw', 'videos')
    if os.path.exists(path_fake):
        archivos_fake = sorted([f for f in os.listdir(path_fake) if f.endswith('.mp4')])
        print(f"📂 Fakes encontrados: {len(archivos_fake)} vídeos en {path_fake}")
        for f in archivos_fake:
            v_id = os.path.splitext(f)[0]  # ej: '000_003'
            instrucciones.append({
                'path': os.path.join(path_fake, f),
                'label': 'fake',
                'tech': 'Deepfakes',
                'id': v_id
            })
    else:
        print(f"❌ No encontrada carpeta de fakes: {path_fake}")

    return pd.DataFrame(instrucciones)

# ==========================================
# 3. EXTRACCIÓN DE FRAMES Y CARAS
# ==========================================
def ejecutar_extraccion(df):
    print("-" * 70)
    print(f"📊 RESUMEN DEL PLAN C0:")
    print(f"Total vídeos: {len(df)}")
    print(df.groupby('label').size())
    print("-" * 70)

    total_imgs = 0
    for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
        target_dir = os.path.join(DESTINY_PATH, 'test', row['label'])
        os.makedirs(target_dir, exist_ok=True)

        nombre_base = f"C0_{row['tech']}_{row['id']}"

        # Saltar si ya está procesado
        if os.path.exists(os.path.join(target_dir, f"{nombre_base}_f14.jpg")):
            continue

        cap = cv2.VideoCapture(row['path'])
        total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_f < FRAMES_PER_VIDEO:
            cap.release()
            continue

        indices = np.linspace(0, total_f - 1, FRAMES_PER_VIDEO, dtype=int)

        for i_frame, pos in enumerate(indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
            ret, frame = cap.read()
            if not ret:
                break
            try:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                face = mtcnn(frame_rgb)
                if face is not None:
                    face_img = face.permute(1, 2, 0).cpu().numpy().astype(np.uint8)
                    file_name = f"{nombre_base}_f{i_frame}.jpg"
                    cv2.imwrite(
                        os.path.join(target_dir, file_name),
                        cv2.cvtColor(face_img, cv2.COLOR_RGB2BGR)
                    )
                    total_imgs += 1
            except:
                continue
        cap.release()

    print(f"\n✨ PROCESO FINALIZADO. Total imágenes para test: {total_imgs}")

# ==========================================
# 4. EJECUCIÓN
# ==========================================
df_test = generar_plan_balanceado()
ejecutar_extraccion(df_test)

c:\TFG\venv_tfg\Lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_path)
c:\TFG\venv_tfg\

✅ Dispositivo: cuda
📂 Reales encontrados: 200 vídeos en C:\TFG\Dataset_C0\original_sequences\youtube\raw\videos
📂 Fakes encontrados: 200 vídeos en C:\TFG\Dataset_C0\manipulated_sequences\Deepfakes\raw\videos
----------------------------------------------------------------------
📊 RESUMEN DEL PLAN C0:
Total vídeos: 400
label
fake    200
real    200
dtype: int64
----------------------------------------------------------------------


100%|██████████| 400/400 [34:01<00:00,  5.10s/it]


✨ PROCESO FINALIZADO. Total imágenes para test: 6000


## Comprobación del dataset
### Balanceo 50/50, caras recortadas con MTCNN y sin data leakage

In [5]:
import os
import pandas as pd

PATH_PROCESADO = r'C:\TFG\Dataset_C0_Procesado\test'

def auditar_dataset_c0(ruta_base):
    print(f"🔍 Iniciando auditoría en: {ruta_base}")
    print("-" * 50)

    if not os.path.exists(ruta_base):
        print(f"❌ ERROR: No existe la ruta {ruta_base}")
        return

    datos = []
    for label in ['real', 'fake']:
        ruta_label = os.path.join(ruta_base, label)
        if not os.path.exists(ruta_label):
            print(f"⚠️ Aviso: No se encuentra la subcarpeta: {label}")
            continue

        archivos = [f for f in os.listdir(ruta_label) if f.endswith('.jpg')]
        for arch in archivos:
            # Ejemplo nombre: C0_Deepfakes_000_003_f0.jpg
            partes = arch.replace('.jpg', '').split('_')
            try:
                tech = partes[1]
                v_id = '_'.join(partes[2:-1])  # puede ser '000' o '000_003'
                datos.append({'archivo': arch, 'label': label, 'id': v_id, 'tech': tech})
            except:
                continue

    df = pd.DataFrame(datos)

    if df.empty:
        print("❌ El dataset está vacío. Verifica que se hayan guardado imágenes.")
        return

    # 1. BALANCEO
    print("📊 BALANCEO DE IMÁGENES:")
    for lab, cant in df['label'].value_counts().items():
        print(f"   - {lab.upper()}: {cant} imágenes ({cant/len(df)*100:.1f}%)")

    # 2. VÍDEOS ÚNICOS
    print("\n🎬 VÍDEOS ÚNICOS POR CLASE:")
    for lab, cant in df.groupby('label')['id'].nunique().items():
        print(f"   - {lab.upper()}: {cant} vídeos únicos")

    # 3. DATA LEAKAGE
    ids_reales = set(df[df['label'] == 'real']['id'])
    ids_fakes  = set(df[df['label'] == 'fake']['id'])
    solapados  = ids_reales.intersection(ids_fakes)
    print("\n🛡️ ANÁLISIS DE DATA LEAKAGE:")
    if len(solapados) == 0:
        print("   ✅ LIMPIO: No hay IDs compartidos.")
    else:
        print(f"   ℹ️ INFO: Hay {len(solapados)} IDs compartidos (normal en FF++ ya que los fakes derivan de los reales).")

    # 4. INTEGRIDAD DE FRAMES
    print("\n📸 INTEGRIDAD DE FRAMES:")
    frames_por_video = df.groupby(['label', 'id']).size()
    incompletos = frames_por_video[frames_por_video < 15]
    if incompletos.empty:
        print("   ✅ PERFECTO: Todos los vídeos tienen 15 frames.")
    else:
        print(f"   ⚠️ AVISO: {len(incompletos)} vídeos tienen menos de 15 frames.")
        print(incompletos)

auditar_dataset_c0(PATH_PROCESADO)

🔍 Iniciando auditoría en: C:\TFG\Dataset_C0_Procesado\test
--------------------------------------------------
📊 BALANCEO DE IMÁGENES:
   - REAL: 3000 imágenes (50.0%)
   - FAKE: 3000 imágenes (50.0%)

🎬 VÍDEOS ÚNICOS POR CLASE:
   - FAKE: 200 vídeos únicos
   - REAL: 200 vídeos únicos

🛡️ ANÁLISIS DE DATA LEAKAGE:
   ✅ LIMPIO: No hay IDs compartidos.

📸 INTEGRIDAD DE FRAMES:
   ✅ PERFECTO: Todos los vídeos tienen 15 frames.
